# 05. Generación y validación

## Información

### Objetivo

Validar el funcionamiento completo del pipeline RAG de ALESSIA, desde la recuperación del contexto hasta la generación de una respuesta utilizando un modelo de lenguaje.

### Objetivos específicos

- Recuperar los fragmentos más relevantes desde el Vector Store.
- Reordenar los resultados mediante el módulo de Reranking.
- Construir el prompt que será enviado al modelo de lenguaje.
- Generar una respuesta utilizando el proveedor LLM configurado.
- Analizar la calidad de la respuesta obtenida.
- Validar la integración de todos los módulos del pipeline RAG.

### Entradas

- Base vectorial persistida en ChromaDB.
- Embeddings generados durante la etapa de indexación.
- Pregunta realizada por el usuario.
- Configuración del proveedor LLM.

### Salidas

- Fragmentos recuperados.
- Fragmentos reordenados por relevancia.
- Prompt generado.
- Respuesta del modelo de lenguaje.
- Validación del funcionamiento del pipeline RAG.

---


## Etapa 1. Ejecución del pipeline RAG

En esta etapa se ejecuta el flujo completo del sistema. A partir de una consulta del usuario, ALESSIA recupera el contexto más relevante, lo reordena mediante el modelo de Reranking, construye el prompt y genera una respuesta utilizando el proveedor LLM configurado.


### 1. Preparación del entorno
Importar los módulos necesarios y se configura el entorno de trabajo para ejecutar el pipeline RAG completo.


In [1]:
from pathlib import Path
import sys

# Agregar la raíz del proyecto al PATH para importar módulos de src
project_root = Path.cwd().parent
sys.path.append(str(project_root))

### 2. Importar módulos del pipeline RAG

In [2]:
from src.processing.vector_store import (
    get_or_create_collection,
)

from src.processing.retriever import retrieve_context

from src.processing.reranker import rerank_chunks

from src.processing.generation import generate_answer

c:\ALURA_AI_Proyect\Challege_Alura_ONE_AI_FOR_TECH\Challenge_Alura_Agente\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 3. Cargar componentes del pipeline RAB

En esta etapa se inicializan los componentes necesarios para ejecutar el flujo completo de ALESSIA.

Se carga la colección persistente de ChromaDB, el modelo utilizado para generar embeddings de las consultas y el modelo CrossEncoder encargado del reordenamiento de resultados.

In [3]:
from sentence_transformers import SentenceTransformer, CrossEncoder

# Ruta del vector store persistente
vector_store_path = project_root / "data" / "vector_store"

# Cargar colección ChromaDB
collection = get_or_create_collection(
    collection_name="alessia_collection",
    vector_store_path=vector_store_path,
)

# Cargar modelo de embeddings
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Cargar modelo CrossEncoder para reranking
reranker_model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6856.75it/s]


### 4. Configuración del proveedor LLM

En esta etapa se valida la configuración del proveedor de lenguaje utilizado por ALESSIA.

Actualmente el sistema utiliza OpenRouter como proveedor principal y el modelo liquid/lfm-2.5-1.2b-instruct:free.

In [4]:
from src.settings import (
    DEFAULT_PROVIDER,
    OPENROUTER_MODEL,
)

print(f"Proveedor activo: {DEFAULT_PROVIDER}")
print(f"Modelo activo: {OPENROUTER_MODEL}")

Proveedor activo: openrouter
Modelo activo: openrouter/auto


### 5. Ejecutar consulta del usuario

En esta etapa se define una consulta que será procesada por el pipeline completo de ALESSIA.

La pregunta será utilizada para recuperar información relevante desde el Vector Store, seleccionar los fragmentos más adecuados mediante Reranking y finalmente generar una respuesta utilizando el modelo de lenguaje configurado.

In [5]:
query = """
¿Qué medidas debe tomar la comunidad ante una emergencia por inundación?
"""

### Recupera fragmentos

In [6]:
retrieved_chunks = retrieve_context(
    query=query,
    collection=collection,
    model=embedding_model,
    k=5,
)

### Reordena fragmentos

In [7]:
reranked_results = rerank_chunks(
    query=query,
    chunks=retrieved_chunks,
    model=reranker_model,
    top_k=3,
)

### 6. Generar la respuesta

Se construye el prompt utilizando la pregunta del usuario y los fragmentos seleccionados durante el proceso de Reranking.

Posteriormente, el prompt es enviado al proveedor de modelos de lenguaje configurado para generar una respuesta fundamentada exclusivamente en el contexto recuperado.

In [8]:
# from openrouter import OpenRouter
# import os

# with OpenRouter(api_key=os.getenv("OPENROUTER_API_KEY")) as client:
#     response = client.chat.send(
#         model=OPENROUTER_MODEL,
#         messages=[
#             {"role": "user", "content": query}
#         ],
#     )

#     print(response.choices[0].message.content)

In [9]:
for i, chunk in enumerate(
    reranked_results,
    start=1,
):
    print("=" * 80)
    print(f"Chunk seleccionado {i}")
    print(
        f"Score: {chunk.metadata['rerank_score']:.4f}"
    )
    print()
    print(chunk.content[:500])

Chunk seleccionado 1
Score: 5.6534

el levantamiento y evaluación de daños y necesidades de la comunidad afectada por una 
emergencia. Uno de los principales instrumentos corresponde al informe ALFA que es de 
aplicación comunal y corresponde a informa ción preliminar de la afectación en una 
emergencia. Un segundo instrumento corresponde a la Ficha Básica de Emergencia (FIBE) 
que recopila información detallada e individualizada de las personas afectadas de un grupo
Chunk seleccionado 2
Score: 4.0092

procesos de la Fase de Respuesta. Los flujos de comunicación entre los organismos, estructuras del 
sistema y niveles territoriales, contribuyen a una adecuada gestión de la emergencia. Por otra parte, 
la entrega de información clara, precisa, objetiva e inclusiva a la comunidad y medios de 
comunicación constituyen parte fundamental de la responsabilidad del municipio y de las 
autoridades respectivas. 
 
6.1.1. Flujos de Comunicación para la Coordinación de la Respuesta.
Chunk selecci

In [11]:
answer = generate_answer(
    question=query,
    context=reranked_results,
)

print(answer)

Hola, vecino.

Con la información disponible, puedo comentarte que los documentos oficiales se centran en los procesos de **evaluación de daños y necesidades** tras una emergencia, así como en los **flujos de comunicación** para coordinar la respuesta entre organismos y con la comunidad.

Se menciona el **informe ALFA** como un instrumento de aplicación comunal que entrega información preliminar sobre la afectación en una emergencia, y la **Ficha Básica de Emergencia (FIBE)** para recopilar información detallada e individualizada de las personas afectadas.

Sin embargo, en la documentación que tengo a mi disposición, **no se detallan medidas específicas que la comunidad deba tomar ante una emergencia por inundación**.

Para poder orientarte de forma más completa, ¿podrías indicarme si tienes alguna pregunta específica relacionada con la información de emergencia o los planes comunales que se mencionan en los documentos? O si tienes acceso a alguna otra información oficial que te gustar